## Imports

In [1]:
import os
import json
import random
import shutil
import tempfile

import torch

from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.9 MB/s eta 0:00:00


In [3]:
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


In [4]:
import re
import ijson
from collections import Counter

from tokenizers import ByteLevelBPETokenizer

## Configuration


In [5]:
# =========================
# PROJECT CONFIGURATION
# =========================

# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

ARCH_TRAIN_TOKENS = 50_000_000
VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01
SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train_supreme.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val_supreme.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# =========================
# TRAINING CONFIGURATION
# =========================


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Checkpoint related functions

In [7]:
BUCKET_NAME = "A-240031/legal-lm-phase1-bucket"
BUCKET_ROOT = f"hf://buckets/{BUCKET_NAME}"

from huggingface_hub import HfFileSystem

fs = HfFileSystem()

In [8]:
def upload_to_bucket(local_path, remote_path):
    remote_path = remote_path.lstrip("/")

    fs.put_file(
        local_path,
        f"{BUCKET_ROOT}/{remote_path}",
    )

    print(f"Uploaded: {remote_path}")

In [9]:
def download_from_bucket(remote_path, local_path):
    remote_path = remote_path.lstrip("/")

    fs.get_file(
        f"{BUCKET_ROOT}/{remote_path}",
        local_path,
    )

    print(f"Downloaded: {remote_path} -> {local_path}")

In [10]:
def upload_checkpoint(tmp_dir, tag):

    remote_dir = f"checkpoints/{tag}"

    for filename in os.listdir(tmp_dir):
        local_path = os.path.join(tmp_dir, filename)

        upload_to_bucket(
            local_path,
            f"{remote_dir}/{filename}",
        )

In [11]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)

        upload_checkpoint(tmp_dir,tag)

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "scheduler.pt",
            "scaler.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        for filename in files:
            try:
                download_from_bucket(
                    f"checkpoints/{tag}/{filename}",
                    os.path.join(checkpoint_dir, filename),
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )

        scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")

        if scheduler is not None and os.path.exists(scheduler_path):
            scheduler.load_state_dict(
                torch.load(
                    scheduler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        scaler_path = os.path.join(checkpoint_dir, "scaler.pt")

        if scaler is not None and os.path.exists(scaler_path):
            scaler.load_state_dict(
                torch.load(
                    scaler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [12]:
from huggingface_hub import hf_hub_download


def download_dataset(name):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        filename=DATASET_FILES[name],
    )

In [13]:
SUPREME_COURT_PATH = download_dataset("supreme_court")

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

In [14]:
def corpus_iterator(path):
    with open(path, "rb") as f:
        for record in ijson.items(f, "item"):
            text = record.get("text")
            if text:
                yield text

In [15]:
tokenizer = ByteLevelBPETokenizer(add_prefix_space=False,lowercase=False)
tokenizer.train_from_iterator(corpus_iterator(SUPREME_COURT_PATH),vocab_size=VOCAB_SIZE,min_frequency=MIN_FREQUENCY,special_tokens=SPECIAL_TOKENS)

In [16]:
tokenizer

Tokenizer(vocabulary_size=32768, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)

In [17]:
def add_custom_legal_tokens(tokenizer, corpus_path, min_frequency=100):
    patterns = [
        re.compile(r"\b(?:Article|ARTICLE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Section|SECTION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Rule|RULE)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\b(?:Regulation|REGULATION)\s+\d+[A-Za-z]*(?:\([0-9A-Za-z]+\))*"),
        re.compile(r"\([a-z]\)"),
        re.compile(r"\([0-9]{1,2}\)"),
        re.compile(r"\((?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2})\)"),
    ]

    counts = Counter()

    for text in corpus_iterator(corpus_path):
        for pattern in patterns:
            counts.update(pattern.findall(text))

    custom_tokens = []

    for token, count in counts.items():
        token_with_space = "Ġ" + token

        if count >= min_frequency and tokenizer.token_to_id(token_with_space) is None:
            custom_tokens.append(token_with_space)

    tokenizer.add_tokens(custom_tokens)

    return custom_tokens, counts

In [18]:
custom_tokens, custom_counts = add_custom_legal_tokens(tokenizer,SUPREME_COURT_PATH,CUSTOM_TOKEN_MIN_FREQUENCY)

print("Custom tokens added:", len(custom_tokens))
print("Final vocabulary size:", tokenizer.get_vocab_size())

Custom tokens added: 1119
Final vocabulary size: 33887


In [19]:
vocab = tokenizer.get_vocab()
print(list(vocab)[:100])

VOCAB_SIZE = len(vocab)

['Ġ113', 'Ġdeficiency', 'Ġreconsideration', 'ĠRANGNATHAN', 'illong', 'ĠKend', 'Ġadding', '224', 'ĠJag', 'Ġinsisting', 'investment', 'Ġprestige', '150', 'Ġsignificance', 'Ġinducing', 'ĠRule 285', 'Ġ364', 'Pri', 'ĠM', 'ĠGAV', 'ĠVessel', 'ĠCEA', 'HOTRA', 'Ġestim', ',--', 'ĠGy', 'Deepak', 'ĠState', 'Ġunavoidable', '228', 'ĠJamshed', 'ographical', 'Ġpermits', 'Ġdonor', 'Ġ308', 'ĠKala', 'Ġprinciple', 'ĠVer', 'Ġplank', 'Ġsentiment', 'Ġdepre', 'Ġoften', 'ordarshan', 'Ġevidently', 'Ġequipments', 'Ġ297', 'Ġcollab', 'Ġbrass', 'Ġwithin', 'ĠJURISDI', 'Ġredetermination', 'ĠHereinafter', 'Ġbleeding', 'nai', 'Ġdev', 'Ġtransl', 'Ġconstra', 'ĠExs', 'Ġimplead', 'avaru', 'since', 'ĠFeb', 'ĠSection 14B', 'ĠDham', 'Ġrather', 'TAX', 'cases', 'Ġignores', 'Ġshells', 'Ġadjusting', 'ĠDattat', 'ught', 'Ġshrine', 'haram', 'Ġ412', 'ĠÌ', 'Ġallotted', 'ĠWilli', 'ĠVijay', 'Ġsouthern', 'Ġminds', 'ĠOfficials', 'Ġdrastically', 'ĠDW', 'ĠHandic', 'Ġspecified', 'ĠAreas', 'Ġcoma', 'rength', 'ĠAla', 'Ġgratification', 'orable'

In [20]:
text = "The Court held that Section 302 applies under Article 21 and Rule 6(2)."

encoding = tokenizer.encode(text)

print("TOKENS:")
print(encoding.tokens)

print("\nDECODED:")
print(repr(tokenizer.decode(encoding.ids)))

print("\nMATCH:")
print(tokenizer.decode(encoding.ids) == text)

TOKENS:
['The', 'ĠCourt', 'Ġheld', 'Ġthat', 'ĠSection', 'Ġ302', 'Ġapplies', 'Ġunder', 'ĠArticle', 'Ġ21', 'Ġand', 'ĠRule', 'Ġ6', '(', '2', ').']

DECODED:
'The Court held that Section 302 applies under Article 21 and Rule 6(2).'

MATCH:
True


## Splitting dataset into train and val set for choosing architecture

In [21]:
import numpy as np
def load_or_create_architecture_data(path):

    try:
        download_from_bucket(
                "architecture_data/arch_train_supreme.bin",
                ARCH_TRAIN_PATH,
            )
            
        download_from_bucket(
                "architecture_data/arch_val_supreme.json",
                ARCH_VAL_PATH,
            )

        print("Loaded architecture data from Hugging Face.")
        return

    except Exception:
        print("Architecture data not found. Creating it...")

    docs = list(corpus_iterator(path))

    split = int(len(docs) * VAL_DOC_FRACTION)

    val_docs = docs[:split]
    train_docs = docs[split:]

    with open(ARCH_TRAIN_PATH, "wb") as f:
        for text in train_docs:
            ids = tokenizer.encode(text).ids
            ids.append(tokenizer.token_to_id("<|eos|>"))
            np.asarray(ids, dtype=np.uint16).tofile(f)

    val_examples = []

    for text in val_docs:
        words = text.split()

        if len(words) < VAL_PROMPT_WORDS + VAL_TARGET_WORDS:
            continue

        for start in range(
            0,
            len(words) - VAL_PROMPT_WORDS - VAL_TARGET_WORDS + 1,
            VAL_PROMPT_WORDS,
        ):
            val_examples.append({
                "prompt": " ".join(words[start:start + VAL_PROMPT_WORDS]),
                "target": " ".join(
                    words[
                        start + VAL_PROMPT_WORDS:
                        start + VAL_PROMPT_WORDS + VAL_TARGET_WORDS
                    ]
                ),
            })

            if len(val_examples) >= VAL_EXAMPLES:
                break

        if len(val_examples) >= VAL_EXAMPLES:
            break

    with open(ARCH_VAL_PATH, "w", encoding="utf-8") as f:
        json.dump(val_examples, f, ensure_ascii=False)

    upload_to_bucket(ARCH_TRAIN_PATH,"architecture_data/arch_train_supreme.bin")

    # Upload mixed architecture validation data
    upload_to_bucket(ARCH_VAL_PATH,"architecture_data/arch_val_supreme.json")

    print("Architecture data created and uploaded.")

In [22]:
load_or_create_architecture_data(SUPREME_COURT_PATH)


(…)architecture_data/arch_train_supreme.bin:   0%|          | 0.00/508M [00:00<?, ?B/s]

Downloaded: architecture_data/arch_train_supreme.bin -> /kaggle/working/architecture_data/arch_train_supreme.bin


(…)/architecture_data/arch_val_supreme.json:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Downloaded: architecture_data/arch_val_supreme.json -> /kaggle/working/architecture_data/arch_val_supreme.json
Loaded architecture data from Hugging Face.


In [23]:
VOCAB_SIZE = tokenizer.get_vocab_size()

D_MODEL = 768
N_LAYERS = 12
N_HEADS = 12
N_KV_HEADS = 4          # GQA: 4 KV heads, 12 query heads
D_FF = 3072
SEQ_LEN = 2048

BATCH_SIZE = 4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 5.0

ARCH_TRAIN_TOKENS = 1_000_000
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

In [24]:
# =========================
# SEQUENTIAL TOKEN DATASET
# =========================
SEQ_LEN = 2048
from torch.utils.data import Dataset, DataLoader
class TokenDataset(Dataset):
    def __init__(self, path, seq_len):
        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len
        self.num_sequences = len(self.tokens) // seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len

        return torch.from_numpy(
            self.tokens[start:end].astype(np.int64)
        )


train_dataset = TokenDataset(
    ARCH_TRAIN_PATH,
    SEQ_LEN,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

TOKENS_PER_STEP = BATCH_SIZE * SEQ_LEN
TOKENS_PER_EPOCH = CHECKPOINT_INTERVAL_TOKENS
TOTAL_TOKENS = min(
    ARCH_TRAIN_TOKENS,
    len(train_dataset) * SEQ_LEN,
)



In [25]:
with open(ARCH_VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

## Model - A1 

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, head_dim, 2).float() / head_dim
            )
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos(),
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin(),
            persistent=False,
        )

    def forward(self, q, k):

        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU Expert
# ============================================================

class SwiGLU(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.up_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.down_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(self.gate_proj(x))
            * self.up_proj(x)
        )


# ============================================================
# Mixture of Experts
# ============================================================
class MoE(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        # Router chooses experts BEFORE any expert is evaluated
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False,
        )

        expert_d_ff = d_ff // top_k

        self.experts = nn.ModuleList([
            SwiGLU(
                d_model,
                expert_d_ff,
            )
            for _ in range(n_experts)
        ])

    def forward(self, x):

        # x: [B, T, D]

        # ------------------------------------------------
        # Batch representation
        # ------------------------------------------------

        batch_repr = x.mean(
            dim=(0, 1)
        )  # [D]

        # ------------------------------------------------
        # ROUTE FIRST
        # ------------------------------------------------

        router_logits = self.router(
            batch_repr
        )  # [E]

        router_probs = F.softmax(
            router_logits,
            dim=-1,
        )

        topk_probs, topk_indices = torch.topk(
            router_probs,
            self.top_k,
            dim=-1,
        )

        # Normalize selected weights
        topk_probs = topk_probs / (
            topk_probs.sum() + 1e-9
        )

        # ------------------------------------------------
        # ONLY NOW evaluate selected experts
        # ------------------------------------------------

        output = 0.0

        for i in range(self.top_k):

            expert_idx = topk_indices[i].item()
            weight = topk_probs[i]

            expert_output = self.experts[
                expert_idx
            ](x)

            output = (
                output
                + weight * expert_output
            )

        # ------------------------------------------------
        # Batch routing loss
        # ------------------------------------------------

        # Router probability distribution for this batch.
        # Used to discourage permanent collapse to
        # a small subset of experts.
        balance_loss = (
            self.balance_alpha
            * self.n_experts
            * torch.sum(
                router_probs
                * router_probs
            )
        )

        return output, balance_loss

# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
    ):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = (
            d_model // n_heads
        )

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=2048,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        repeat_factor = (
            self.n_heads
            // self.n_kv_heads
        )

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(
            attn_out
        )


# ============================================================
# Decoder Block with MoE
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(
            d_model
        )

        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(
            d_model
        )

        self.moe = MoE(
            d_model=d_model,
            d_ff=d_ff,
            n_experts=n_experts,
            top_k=top_k,
            balance_alpha=balance_alpha,
        )

    def forward(self, x):

        # Attention
        x = x + self.attention(
            self.norm1(x)
        )

        # MoE FFN
        moe_output, balance_loss = self.moe(
            self.norm2(x)
        )

        x = x + moe_output

        return x, balance_loss


# ============================================================
# Decoder-only MoE Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,

        # MoE
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                n_experts=n_experts,
                top_k=top_k,
                balance_alpha=balance_alpha,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(
            d_model
        )

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.Embedding):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.RMSNorm):

            nn.init.ones_(
                module.weight
            )

    def forward(
        self,
        input_ids,
        return_aux_loss=False,
    ):

        x = self.token_embedding(
            input_ids
        )

        total_balance_loss = x.new_zeros(
            ()
        )

        for block in self.blocks:

            x, balance_loss = block(x)

            total_balance_loss = (
                total_balance_loss
                + balance_loss
            )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        if return_aux_loss:

            return (
                logits,
                total_balance_loss,
            )

        return logits

In [27]:
d_model    = 512
n_layers   = 12
n_heads    = 8
n_kv_heads = 4
d_ff       = 2048
SEQ_LEN  = 2048

n_experts  = 8
top_k      = 2

In [28]:
model = EfficientDecoderLM(
    vocab_size=VOCAB_SIZE,
    d_model=d_model,
    n_heads=n_heads,
    n_kv_heads=n_kv_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_seq_len=SEQ_LEN,
).to(DEVICE)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

Parameters: 177.84M


In [29]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1,
)


In [30]:
import time
import torch
import torch.nn.functional as F
from bert_score import score as bert_score


def train(
    model,
    train_loader,
    optimizer,
    val_data,
    train_tokens,
    tag,
    val_interval_tokens=1_000_000,
    resume=False,
    resume_tag=None,
):
    model.train()

    tokens_seen = 0
    global_step = 0
    epoch = 0
    next_val_tokens = val_interval_tokens

    if resume:
        if resume_tag is None:
            raise ValueError("resume_tag must be provided when resume=True")

        state = load_checkpoint(
            tag=resume_tag,
            model=model,
            optimizer=optimizer,
        )

        tokens_seen = state["tokens_seen"]
        global_step = state["global_step"]
        epoch = state["epoch"]

        next_val_tokens = (
            (tokens_seen // val_interval_tokens) + 1
        ) * val_interval_tokens

        print(
            f"Resumed from {resume_tag} | "
            f"Tokens: {tokens_seen:,} | "
            f"Step: {global_step}"
        )

    start_time = time.perf_counter()

    while tokens_seen < train_tokens:

        for input_ids in train_loader:

            if tokens_seen >= train_tokens:
                break

            input_ids = input_ids.to(DEVICE, non_blocking=True)

            inputs = input_ids[:, :-1]
            targets = input_ids[:, 1:]
        
            optimizer.zero_grad(
                set_to_none=True
            )
        
            logits, balance_loss = model(
                inputs,
                return_aux_loss=True,
            )
        
            lm_loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    model.vocab_size,
                ),
                targets.reshape(-1),
            )
        
            loss = lm_loss + balance_loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            batch_tokens = input_ids.numel()
            tokens_seen += batch_tokens
            global_step += 1

            torch.cuda.synchronize()

            elapsed = time.perf_counter() - start_time
            tokens_per_sec = tokens_seen / elapsed

            print(
                f"Step {global_step:5d} | "
                f"Tokens {tokens_seen:,}/{train_tokens:,} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.2f}s | "
                f"Tokens/s {tokens_per_sec:,.0f}"
            )
            
            # inside train(), replace the generation/BERTScore validation block with:
            
            if tokens_seen >= next_val_tokens:
            
                model.eval()
            
                val_loss = 0.0
                val_tokens = 0
            
                with torch.no_grad():
                    for example in val_data:
            
                        prompt_ids = tokenizer.encode(example["prompt"]).ids
                        target_ids = tokenizer.encode(example["target"]).ids
            
                        input_ids = torch.tensor(
                            [prompt_ids + target_ids],
                            dtype=torch.long,
                            device=DEVICE,
                        )
            
                        logits = model(input_ids,return_aux_loss=False)
            
                        prompt_len = len(prompt_ids)
            
                        target_logits = logits[:, prompt_len - 1:-1, :]
                        target = input_ids[:, prompt_len:]
            
                        loss = F.cross_entropy(
                            target_logits.reshape(-1, target_logits.size(-1)),
                            target.reshape(-1),
                            reduction="sum",
                        )
            
                        val_loss += loss.item()
                        val_tokens += target.numel()
            
                val_loss /= val_tokens
            
                print(
                    f"Validation @ {tokens_seen:,} tokens | "
                    f"Val Loss: {val_loss:.4f}"
                )
            
                model.train()
            
                save_checkpoint(
                    tag=tag,
                    model=model,
                    optimizer=optimizer,
                    tokens_seen=tokens_seen,
                    global_step=global_step,
                    epoch=epoch,
                )
            
                print(f"Checkpoint saved: {tag}\n")
            
                next_val_tokens += val_interval_tokens

    total_time = time.perf_counter() - start_time

    print("\nTraining complete.")
    print(f"Tokens seen: {tokens_seen:,}")
    print(f"Total time: {total_time:.2f}s")
    print(f"Average tokens/s: {tokens_seen / total_time:,.0f}")

In [31]:
train(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    val_data=val_data,
    train_tokens=125_000_000,
    tag="decoder_moe_125M",
    resume = True,
    resume_tag = "decoder_moe_125M"
)

(…)et/checkpoints/decoder_moe_125M/model.pt:   0%|          | 0.00/712M [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_moe_125M/model.pt -> /tmp/tmpwf5y5z96/checkpoints/decoder_moe_125M/model.pt


(…)heckpoints/decoder_moe_125M/optimizer.pt:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_moe_125M/optimizer.pt -> /tmp/tmpwf5y5z96/checkpoints/decoder_moe_125M/optimizer.pt


(…)heckpoints/decoder_moe_125M/rng_state.pt:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_moe_125M/rng_state.pt -> /tmp/tmpwf5y5z96/checkpoints/decoder_moe_125M/rng_state.pt


(…)nts/decoder_moe_125M/training_state.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_moe_125M/training_state.json -> /tmp/tmpwf5y5z96/checkpoints/decoder_moe_125M/training_state.json
Resumed from decoder_moe_125M | Tokens: 125,001,728 | Step: 15259

Training complete.
Tokens seen: 125,001,728
Total time: 0.00s
Average tokens/s: 96,377,590,686,053


In [32]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

test.json: 0.00B [00:00, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [33]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


In [34]:
@torch.no_grad()
def generate_single(
    prompt,
    model,
    tokenizer,
    device,
    target_words=128,
    temperature=0.8,
    top_p=0.9,
):
    model.eval()

    max_input_len = SEQ_LEN - MAX_GENERATION_TOKENS

    prompt_ids = tokenizer.encode(prompt).ids[:max_input_len]

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device,
    )

    generated_ids = []

    # ========================================================
    # Autoregressive generation
    # ========================================================

    for _ in range(MAX_GENERATION_TOKENS):

        # ----------------------------------------------------
        # Run the ENTIRE sequence every time
        # ----------------------------------------------------

        logits = model(input_ids,return_aux_loss = False)

        next_logits = (
            logits[:, -1, :]
            / temperature
        )

        # ----------------------------------------------------
        # Top-p sampling
        # ----------------------------------------------------

        sorted_logits, sorted_indices = torch.sort(
            next_logits,
            descending=True,
            dim=-1,
        )

        sorted_probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove = cumulative_probs > top_p

        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False

        sorted_logits[remove] = float("-inf")

        probs = torch.softmax(
            sorted_logits,
            dim=-1,
        )

        sampled = torch.multinomial(
            probs,
            1,
        )

        next_token = torch.gather(
            sorted_indices,
            -1,
            sampled,
        )

        # ----------------------------------------------------
        # Store generated token
        # ----------------------------------------------------

        generated_ids.append(
            next_token.item()
        )

        # ----------------------------------------------------
        # Stop at 128 words
        # ----------------------------------------------------

        text = tokenizer.decode(generated_ids)

        if len(text.split()) >= target_words:
            break

        # ----------------------------------------------------
        # Append token to the FULL sequence
        # ----------------------------------------------------

        input_ids = torch.cat(
            [
                input_ids,
                next_token,
            ],
            dim=1,
        )

    # ========================================================
    # Final prediction
    # ========================================================

    text = tokenizer.decode(generated_ids)

    text = " ".join(
        text.split()[:target_words]
    )

    return text

In [35]:
MAX_GENERATION_TOKENS = 256

In [36]:

predictions = {}

for sample_id, sample in test_data.items():

    prediction = generate_single(
        prompt=sample["text"],
        model=model,
        tokenizer=tokenizer,
        device=DEVICE,
        target_words=128,
        temperature=0.8,
        top_p=0.9,
    )

    predictions[sample_id] = {
        "text": prediction
    }

    print(f"Finished {sample_id}")

Finished 1
Finished 2
Finished 3
Finished 4
Finished 5
Finished 6
Finished 7
Finished 8
Finished 9
Finished 10
Finished 11
Finished 12
Finished 13
Finished 14
Finished 15
Finished 16
Finished 17
Finished 18
Finished 19
Finished 20
Finished 21
Finished 22
Finished 23
Finished 24
Finished 25
Finished 26
Finished 27
Finished 28
Finished 29
Finished 30
Finished 31
Finished 32
Finished 33
Finished 34
Finished 35
Finished 36
Finished 37
Finished 38
Finished 39
Finished 40
Finished 41
Finished 42
Finished 43
Finished 44
Finished 45
Finished 46
Finished 47
Finished 48
Finished 49
Finished 50
Finished 51
Finished 52
Finished 53
Finished 54
Finished 55
Finished 56
Finished 57
Finished 58
Finished 59
Finished 60
Finished 61
Finished 62
Finished 63
Finished 64
Finished 65
Finished 66
Finished 67
Finished 68
Finished 69
Finished 70
Finished 71
Finished 72
Finished 73
Finished 74
Finished 75
Finished 76
Finished 77
Finished 78
Finished 79
Finished 80
Finished 81
Finished 82
Finished 83
Finished 84
F

In [37]:
import zipfile
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:
    zipf.write(
        PREDICTION_PATH,
        arcname="prediction.json"
    )

print("Codabench submission created:")
print(SUBMISSION_ZIP)

prediction.json saved:
/kaggle/working/cs781_phase1/prediction.json
Codabench submission created:
/kaggle/working/cs781_phase1/submission.zip
